[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DL_4_Physics/PINNs.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Physics-Informed Neural Networks

The flagship idea of deep learning *for physics*: instead of fitting data alone, make the network satisfy the **differential equation itself** — autograd computes the derivatives, the PDE residual becomes a loss term, and physics becomes a regularizer that lets you learn from absurdly little data.

## 1. Pre-requisites

- [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb) — autograd, training loops.
- An ODE/PDE course helps but the two equations used here are self-contained.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 2 — *The PDE-as-Loss Idea* (~35 min)
**Goal:** use autograd to differentiate the network w.r.t. its INPUTS; solve an ODE with zero data.
**Builds on:** [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb). &nbsp; **Feeds into:** Session 2 (a real boundary-value problem).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The PDE-as-Loss Idea</b></summary>

**Timing (~35 min).** 10 min autograd's second job · 10 min the residual as a loss · 10 min the demo · 5 min honest positioning.

**Open with the reframing, because it is one line and it is the whole workshop.** Backprop differentiates the loss with respect to **weights**. But autograd is a general derivative machine — it will just as happily differentiate the network's **output with respect to its input**. For a network $u_\theta(t)$ that gives you $u'(t)$ and $u''(t)$ **exactly**, at any point, with no finite differences and no discretisation error. **Autograd was always able to do this; nobody was asking.**

**Then the move that follows immediately.** If physics says $u' = -\lambda u$, you can *penalise the network for violating it* at any set of points you like: $\mathcal{L}_{\text{phys}} = \frac1N\sum_i (u'(t_i) + \lambda u(t_i))^2$. **The equation becomes training data — infinite, free, and exact.** Sample new collocation points every step and you never run out.

**Walk the four lines of the loop and name each one, since this is the entire method.** Sample collocation points with `requires_grad=True`; evaluate the network; call `torch.autograd.grad(u, t, ..., create_graph=True)`; square the residual. **The `create_graph=True` is load-bearing** — without it the derivative is a constant with respect to the weights and the physics loss has no gradient. Students get exactly one confusing afternoon from omitting it.

**Emphasise that no solution data is used at all here.** The only anchor is the initial condition $u(0) = 1$. Everything else comes from the equation. Ask the room what the network would learn with the physics term but *no* initial condition — the answer is any member of the family $Ce^{-\lambda t}$, including $C = 0$, which is why the IC term is not optional. **The residual selects a family; the boundary conditions select a member.**

**Set expectations for the accuracy honestly, before it appears.** The demo reaches a max error of $5.9\times10^{-3}$ — about **0.6%**. That is a fine sanity check and it is **not competitive with a numerical solver**: a Runge–Kutta integrator reaches $10^{-10}$ on this ODE in microseconds. **PINNs are not a better way to solve ODEs you can already solve**, and a room that leaves believing otherwise has learned something false.

**So state where PINNs actually win, since that is the honest sales pitch.** High dimensions, where mesh-based solvers die of the curse of dimensionality. **Inverse problems**, where a physical constant is unknown and becomes a `nn.Parameter` trained by the same loss. And **data assimilation** — sparse noisy measurements plus a governing equation, which is exactly Session 2. The forward-solve demo is a mechanism check, not a use case.

**Close by naming the framing that ties this to the rest of the curriculum.** A PINN is regularisation where the regulariser is a *differential equation* instead of an L2 penalty. [Training Dynamics](../Intro_Mach_Learn/Training_Dynamics.ipynb) argued that regularisation is a thumb on the scale toward simple functions; **here the thumb is Newton's**, and it is an infinitely informative prior in a way weight decay never is.
</details>

## 2. Autograd's Second Job

💡 **Intuition.** Backprop differentiates the loss w.r.t. *weights*. But autograd is general: it can just as happily differentiate the network's **output w.r.t. its input** — giving us $u'(t)$, $u''(t)$ for a network $u_\theta(t)$, exactly, no finite differences. So if physics says $u' = -\lambda u$, we can *penalize the network for violating it* at any set of collocation points: $\mathcal{L}_{physics} = \frac{1}{N}\sum_i (u'(t_i) + \lambda u(t_i))^2$. The equation becomes training data — infinite, free, and exact.

In [2]:
# Warm-up: solve u' = -1.5 u, u(0)=1 with NO solution data at all
lam = 1.5
net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.Adam(net.parameters(), lr=5e-3)

for step in range(2000):
    t = torch.rand(64, 1, requires_grad=True) * 3      # collocation points in [0, 3]
    u = net(t)
    du = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    loss_phys = ((du + lam * u)**2).mean()             # the ODE residual
    u0 = net(torch.zeros(1, 1))
    loss_ic = (u0 - 1.0)**2                            # initial condition
    loss = loss_phys + loss_ic.squeeze()
    opt.zero_grad(); loss.backward(); opt.step()

tt = torch.linspace(0, 3, 200)[:, None]
with torch.no_grad():
    u_hat = net(tt).squeeze()
u_true = np.exp(-lam * tt.squeeze().numpy())
plt.figure(figsize=(7, 2.6))
plt.plot(tt, u_true, "k--", label="analytic  $e^{-1.5t}$")
plt.plot(tt, u_hat, label="PINN (trained on the EQUATION, zero data)")
plt.legend(); plt.tight_layout(); plt.show()
print(f"max error vs analytic solution: {np.abs(u_hat.numpy() - u_true).max():.2e}")

max error vs analytic solution: 5.87e-03


/tmp/ipykernel_2058678/4044410526.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


**What just happened.** A network learned $e^{-1.5t}$ to a maximum error of **$5.9\times10^{-3}$** — about 0.6% — having **never seen a single value of the solution**. The only inputs were the equation $u' = -1.5u$ and the initial condition $u(0) = 1$.

**The mechanism is one line: `torch.autograd.grad(u, t, ...)`.** Backprop differentiates the loss with respect to *weights*; here we differentiate the *output* with respect to the *input*, getting $u'(t)$ **exactly** — no finite differences, no step size, no discretisation error. **Autograd could always do this; it is simply not what it is normally asked for.**

**Note `create_graph=True`, because omitting it is the classic first bug.** It keeps the derivative computation inside the graph, so the physics loss has a gradient with respect to the weights. Without it, `du` is treated as a constant, `loss_phys.backward()` contributes nothing, and the network trains on the initial condition alone — silently, with no error message. **Every PINN implementation lives or dies on that flag.**

**The initial-condition term is not decoration either.** The residual $u' + \lambda u = 0$ is satisfied by the *entire family* $Ce^{-\lambda t}$ — including $C = 0$, which is a perfect, useless solution the optimiser will happily find. **The residual selects a family; the boundary condition selects a member.** Drop `loss_ic` and the network collapses to zero.

**Now the honest reading of $5.9\times10^{-3}$, because it is not as good as it sounds.** The solution has magnitude 1, so this is 0.6% error after 2,000 optimisation steps. **A fourth-order Runge–Kutta integrator solves this ODE to $10^{-10}$ in microseconds** — five thousand times more accurate, and thousands of times faster. **PINNs are not a better way to solve ODEs you can already solve**, and this demo should not be read as a claim that they are.

**Which makes it important to say where PINNs genuinely win.** *High dimensions*, where mesh-based solvers face the curse of dimensionality and a network does not. *Inverse problems*, where a physical constant is unknown — make $\lambda$ a `nn.Parameter` and the same loss estimates it from data. And *data assimilation*: sparse noisy measurements plus a governing equation, which is Session 2's subject. **This cell is a mechanism check, not a use case.**

**One structural observation worth making while the code is on screen.** Collocation points are drawn **fresh at random each step** — `torch.rand(64,1)*3`. So the training set is effectively infinite and the network never sees the same point twice. **There is no overfitting in the usual sense**, because the "data" is generated on demand from an equation. That is a genuinely different regime from every other workshop in the ML track.

**And note what the loss is not telling you.** The physics loss measures *residual*, not *error*. A network can have a small residual and still be the wrong solution — near-solutions of an ODE can drift far from the true one over a long interval. **Low residual is necessary and not sufficient**, which is why this cell compares against the analytic solution rather than reporting the loss.

---
### 🕐 Session 2 of 2 — *A Real Problem: the Damped Oscillator from 6 Points* (~40 min)
**Goal:** combine sparse noisy data with physics; watch the physics term rescue the fit.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: A Real Problem — the Damped Oscillator</b></summary>

**Timing (~40 min).** 8 min the setup · 12 min the two fits · 10 min what the demo actually proves · 10 min loss weighting and inverse problems.

**Frame the session as the honest selling point.** Session 1 solved an ODE nobody needed solving. **The real case for PINNs is the *combination*:** a handful of noisy measurements cannot pin down a wiggly function, but they can pin down the **constants** of a function that physics has already shaped. Loss = data misfit + PDE residual, and the physics term acts as a prior with effectively infinite information content.

**Have the room predict the plain network's behaviour before running.** Six points, a 3-layer network with about 8,600 parameters, an oscillation at $\omega = 8$ rad/s. **The network has vastly more capacity than the data constrains**, so between and beyond the points it will do whatever the initialisation and optimiser happen to prefer. Ask what "whatever" looks like: a smooth curve through six dots, with no oscillation at all.

**Then read the RMSE honestly, because the plain baseline is worse than trivial.** Plain NN 0.675, PINN 0.027 — a 25× gap. But compute the RMS of the true signal: about **0.50**. **The plain network is worse than predicting zero everywhere.** That is not a subtle failure; the fit is actively harmful. Say so, because "25× better than the baseline" undersells what actually happened.

**Now the caveat that must not be skipped, and it changes the headline.** The PINN loss includes **both initial conditions**: $u(0) = 1$ and $u'(0) = 0$. A second-order linear ODE with two initial conditions has a **unique solution** — it *is* `analytic`, exactly. **So the ODE plus the ICs already determine the answer completely, and the six data points are redundant.** The label "6 points + the ODE" credits the data for work the physics did alone.

**Turn that into the session's best experiment rather than an embarrassment.** Re-run with `t_data` reduced to zero points and the physics weight unchanged: the PINN should still recover the solution. Then remove the IC terms and keep the six points: now the data genuinely selects the member of the solution family, and the demo means what it claims. **Two runs, and the room learns exactly which ingredient is doing the work** — which is the whole point of an ablation.

**Use the plain-vs-PINN contrast for the claim it does support.** Same architecture, same data, same optimiser, same steps. **The only difference is a term in the loss**, and the fits are qualitatively different. That is a clean demonstration that physics-as-regulariser works; it is just not a demonstration that six points were sufficient.

**Then cover the practicalities, since the notebook flags them and they are where students get stuck.** The physics weight of $10^{-3}$ is doing real balancing: the residual involves $\omega^2 = 64$, so its natural scale is orders of magnitude above the data misfit, and an unweighted sum lets the physics term swamp the data entirely. **Loss weighting is the central practical difficulty of PINNs** — there are whole papers on adaptive schemes.

**Close on inverse problems, because that is where PINNs are genuinely unmatched.** Make $\omega$ a `nn.Parameter` and the same loss estimates it from data — no derivative estimation, no spectral fitting, just gradient descent on a residual. **Recovering a physical constant from six noisy points is something no ODE solver does at all**, and it is worth ending the workshop on the capability that has no classical competitor.
</details>

## 3. Data + Physics

💡 **Intuition.** The honest selling point of PINNs is the *combination*: a handful of noisy measurements can't pin down a wiggly function — but they can pin down the **constants** (amplitude, phase) of a function the physics already shapes. Loss = data misfit + PDE residual; the physics term acts as an infinitely-informative prior. Watch a plain network hallucinate between 6 points while the PINN interpolates *and extrapolates* correctly.

In [3]:
# damped oscillator: u'' + 2ζω u' + ω² u = 0,  ω=8, ζ=0.05
w0, zeta = 8.0, 0.05
def analytic(t):
    wd = w0 * np.sqrt(1 - zeta**2)
    return np.exp(-zeta*w0*t) * np.cos(wd*t)

t_data = torch.tensor([[0.05], [0.35], [0.61], [0.92], [1.2], [1.53]])   # SIX points
u_data = torch.tensor(analytic(t_data.numpy())) + 0.02*torch.randn(6, 1)

def make_net():
    torch.manual_seed(3)
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))

def train_net(physics_weight, steps=4000):
    net = make_net()
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for s in range(steps):
        loss = ((net(t_data.float()) - u_data.float())**2).mean()
        if physics_weight > 0:
            t = torch.rand(128, 1, requires_grad=True) * 2.0
            u = net(t)
            du = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
            d2u = torch.autograd.grad(du, t, torch.ones_like(du), create_graph=True)[0]
            resid = d2u + 2*zeta*w0*du + w0**2 * u
            loss = loss + physics_weight * (resid**2).mean()
            # initial conditions u(0)=1, u'(0)=0 anchor the family member
            t0 = torch.zeros(1, 1, requires_grad=True)
            u0 = net(t0)
            du0 = torch.autograd.grad(u0, t0, torch.ones_like(u0), create_graph=True)[0]
            loss = loss + (u0 - 1)**2 + du0**2
        opt.zero_grad(); loss.backward(); opt.step()
    return net

net_plain = train_net(0.0)
net_pinn  = train_net(1e-3)

tt = torch.linspace(0, 2, 400)[:, None]
with torch.no_grad():
    up, ui = net_plain(tt).squeeze(), net_pinn(tt).squeeze()
truth = analytic(tt.squeeze().numpy())

plt.figure(figsize=(9, 3))
plt.plot(tt, truth, "k--", linewidth=1, label="true solution")
plt.plot(tt, up, label="plain NN: 6 points, hallucinated physics")
plt.plot(tt, ui, label="PINN: 6 points + the ODE")
plt.plot(t_data, u_data, "ro", markersize=6, label="the six measurements")
plt.legend(fontsize=8); plt.title("physics as a prior: same data, radically different fits")
plt.tight_layout(); plt.show()
print(f"RMSE  plain NN {np.sqrt(np.mean((up.numpy()-truth)**2)):.3f}   PINN {np.sqrt(np.mean((ui.numpy()-truth)**2)):.3f}")

RMSE  plain NN 0.675   PINN 0.027


/tmp/ipykernel_2058678/925058491.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Same architecture, same six noisy points, same optimiser, same 4,000 steps. **The only difference is one term in the loss**, and the results are:

| model | RMSE vs truth |
|---|---|
| plain NN | **0.675** |
| PINN | **0.027** |

**A 25× gap — but the plain baseline is worse than trivial, which the ratio hides.** The true signal $e^{-0.4t}\cos(7.99t)$ on $[0,2]$ has RMS about **0.50**. So a model that predicted **zero everywhere** would score 0.50, and the plain network scores 0.675. **It is not merely uninformative; it is actively worse than silence.** Six points cannot constrain an 8,600-parameter network, and between them it invents a smooth curve with no oscillation at all.

**Now the caveat that changes the headline, and it should not be skipped.** The PINN's loss includes **both initial conditions** — $u(0) = 1$ and $u'(0) = 0$ — alongside the ODE residual. A second-order linear ODE with two initial conditions has a **unique solution**, and that solution *is* `analytic`. **So the physics alone already determines the answer completely; the six data points are redundant.** The plot's label "6 points + the ODE" credits the data for work the equation did by itself.

**That is worth turning into the session's best experiment rather than glossing over.** Run `train_net(1e-3)` with `t_data` emptied — the PINN should recover the solution anyway. Then run it with the IC terms deleted and the six points kept: now the data genuinely picks the member of the two-parameter solution family, and the demo means exactly what it claims. **Two ablations, and the room learns which ingredient carries the result.**

**What the comparison *does* establish is still real and worth stating precisely.** Adding a differential equation to the loss changes an unusable fit into an accurate one, at identical capacity, data, and compute. **Physics-as-regulariser works.** The claim it does not establish is "six measurements were sufficient" — that requires the ablation above.

**Look at the extrapolation region beyond $t = 1.53$, because it is the most convincing part of the figure.** The last data point is at 1.53 and the plot runs to 2.0. The plain network drifts off immediately — it has no reason not to. The PINN keeps oscillating correctly, because the **residual is enforced at collocation points across the whole interval**, data or no data. **The physics term supervises where the data is silent**, which is the property that makes PINNs interesting for forecasting and gap-filling.

**The weight $10^{-3}$ is doing real work and is the hardest thing to set in practice.** The residual contains $\omega^2 u = 64u$, so its natural magnitude is orders above the data misfit; an unweighted sum would let physics swamp the six points entirely. **Loss weighting is the central practical difficulty of PINNs** — there is a literature of adaptive schemes for exactly this, and 1e-3 here was found by tuning, not derived.

**Finally, the capability with no classical competitor, which the notebook rightly flags.** Make $\omega$ an `nn.Parameter` and the identical loss **estimates the physical constant from the data**. No derivative estimation, no spectral fitting — gradient descent on a residual. **An ODE solver cannot do this at all**, and inverse problems, not forward solves, are where PINNs genuinely earn their place.

**Practicalities worth a slide:** weighting the loss terms is the art (residual and data scales differ — here 1e-3 balances them); stiff/high-frequency problems need Fourier feature inputs or curriculum in time; and PINNs also run *inverse* problems — make $\omega$ a `nn.Parameter` and the same loss estimates the physical constant from data. Try it: you should recover $\omega \approx 8$.

## 4. Conclusion

Autograd differentiates through inputs, so equations become losses, physics becomes a prior, and six noisy points suffice where hundreds were needed. This is the core move of scientific machine learning.

---
## Where next

- [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb) — the autograd machinery.
- [Uncertainty in ML](../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — how much to trust the extrapolation.
- [Kernel Methods](../Intro_Mach_Learn/Kernel_Methods.ipynb) — the classical way of encoding priors, for contrast.